# T8 — head Bi-GRU/Bi-LSTM/CNN đọc token PhoBERT, chạy trên GPU Colab

PhoBERT vẫn **đóng băng**. Trên CPU/MPS ở máy chính, một epoch của head này đo được
**23 phút (CPU)** / **8,5 phút (MPS)** — quá chậm cho `patience=8`. Notebook này chạy
đúng cùng mã nguồn trên GPU T4 miễn phí của Colab, nơi `nn.GRU`/`nn.LSTM` dùng kernel
cuDNN nên nhanh hơn nhiều.

## Việc phải làm **trước khi mở notebook này**

Upload 12 tệp cache sau lên Google Drive, vào **một thư mục** (mặc định notebook
này giả định `MyDrive/vietjobs_cache/` — sửa `DRIVE_CACHE` ở cell dưới nếu khác):

| Tệp | Cỡ |
|---|---|
| `train-raw-len256-tok.f16.npy` | 12,88 GB |
| `train-raw-len256-mask.npy` | 8,39 MB |
| `train-raw-len256-tok.f16.json` | vài trăm B |
| `train-raw-len256.npy` | 100,65 MB (bản gộp, để test canh) |
| `dev-raw-len256-tok.f16.npy` | 1,43 GB |
| `dev-raw-len256-mask.npy` | 0,93 MB |
| `dev-raw-len256-tok.f16.json` | vài trăm B |
| `dev-raw-len256.npy` | 11,17 MB |
| ...và 4 tệp tương tự cho họ `masked` (`train-masked-*`, `dev-masked-*`) | |

Tổng ≈ 28,9 GB. Đây là các tệp `artifacts/embeddings/*-tok.f16.npy` /
`*-mask.npy` / `*.json` / `*.npy` (bản gộp) đã có sẵn trên máy chính — không
phải tính lại gì.

## Notebook làm gì

1. Clone nhánh `t8-rnn-colab` (đã có sẵn mã T8.1/T8.2).
2. Tải `data/raw/VietJobs.csv` từ HuggingFace, xác nhận sha256, `dataset build`
   lại splits — **tất định cùng seed**, nên giống byte-với-byte máy chính.
3. Chép cache từ Drive vào đĩa cục bộ của Colab (đọc thẳng qua FUSE mount chậm).
4. Chạy `pytest` canh cache đúng, rồi 5 lần huấn luyện `--device cuda`.
5. Đóng gói kết quả, lưu lại Drive để mang về máy chính.

In [ ]:
!nvidia-smi

## 1. Clone mã nguồn

In [ ]:
!git clone -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
!git log --oneline -3

## 2. Cài phụ thuộc

**Không** cài lại `torch` — bản CUDA sẵn có của Colab phải giữ nguyên;
`requirements-dl.txt` tự ghi rõ điều này ("Trên Colab/Kaggle thì bỏ ghim torch").

In [ ]:
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" "huggingface_hub"
!pip install -q -e . --no-deps

## 3. Tải dữ liệu gốc và dựng lại splits (tất định)

`data/README.md` cam kết splits byte-với-byte giống nhau trên mọi máy, cùng
`SPLIT_SEED`. Bước dưới xác nhận đúng sha256 trước khi build — sai một byte của
tệp gốc thì cache trên Drive sẽ **không còn khớp thứ tự dòng** với split mới.

In [ ]:
import pathlib, shutil, hashlib
from huggingface_hub import hf_hub_download

EXPECTED_SHA256 = "85862b06fda4e814fe0c1d8622f173d189c92758345f77df16d1232d0c49d477"

p = hf_hub_download("dinhieufam/VietJobs", "VietJobs.csv", repo_type="dataset",
                    cache_dir="data/raw/.cache/huggingface")
pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)
shutil.copy(p, "data/raw/VietJobs.csv")

h = hashlib.sha256(pathlib.Path("data/raw/VietJobs.csv").read_bytes()).hexdigest()
assert h == EXPECTED_SHA256, f"sha256 lệch: {h} != {EXPECTED_SHA256} — DỪNG, đừng build"
print("sha256 khớp, an toàn để build:", h)

In [ ]:
%env PYTHONPATH=src
# Tách từ 47.707 tin ~ 13 phút trên CPU của Colab, không dùng GPU ở bước này.
!python -m vietjobs.dataset build

In [ ]:
import json
m = json.load(open("data/processed/manifest.json"))
assert m["splits"]["train"]["rows"] == 34354
assert m["splits"]["dev"]["rows"] == 3812
assert m["splits"]["test"]["rows"] == 9541
print("manifest khớp máy chính:", m["splits"])

## 4. Mount Drive, chép cache về đĩa cục bộ

Chép, **không đọc thẳng qua Drive mount** — memmap truy cập ngẫu nhiên (fancy
indexing theo batch) trên FUSE rất chậm; đĩa cục bộ của Colab thì không.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_CACHE = "/content/drive/MyDrive/vietjobs_cache"  # sửa nếu bạn để chỗ khác

In [ ]:
import pathlib, shutil, time

FILES = []
for split in ("train", "dev"):
    for family in ("raw", "masked"):
        stem = f"{split}-{family}-len256"
        FILES += [f"{stem}-tok.f16.npy", f"{stem}-mask.npy",
                  f"{stem}-tok.f16.json", f"{stem}.npy"]

dst = pathlib.Path("artifacts/embeddings")
dst.mkdir(parents=True, exist_ok=True)
src_dir = pathlib.Path(DRIVE_CACHE)

for name in FILES:
    src, out = src_dir / name, dst / name
    if not src.exists():
        raise FileNotFoundError(f"thiếu {src} trên Drive — xem danh sách 12 tệp ở đầu notebook")
    t0 = time.time()
    shutil.copy(src, out)
    print(f"  {name}  {out.stat().st_size/1e9:6.2f} GB  {time.time()-t0:5.1f}s")
print("xong, tổng:", sum((dst / n).stat().st_size for n in FILES) / 1e9, "GB")

## 5. Canh cache đúng trước khi huấn luyện

`tests/test_dl_encode.py` so `mean(cache token)` với cache gộp đã có — nếu cache
chép lên bị hỏng hay lệch thứ tự dòng, test này báo ngay thay vì để 3 giờ huấn
luyện chạy trên dữ liệu sai. `tests/test_dl_heads.py` canh riêng phần kiến trúc.

In [ ]:
!pip install -q pytest
!python -m pytest -q tests/test_dl_encode.py tests/test_dl_heads.py

## 6. Năm lần chạy — T8.3, T8.4, T8.5

Mốc so sánh (đo trên máy chính, CPU): `dl-cat-ce-cpu-0920` macro-F1 **0,6030**,
`dl-cat-focal-cpu-0920` **0,5972**, `dl-sal-cpu-0920` MAE **4,13 tr**. Ngưỡng cải
thiện thật: Δ > **0,015** (macro-F1) — nhiễu chạy lại cùng cấu hình đã đo 0,0017.

Mỗi lệnh in tiến độ theo epoch vào `artifacts/<run_id>/history.jsonl`; để chạy
hết cả năm lần, đừng đóng tab trong lúc chạy (Colab miễn phí ngắt khi không có
tương tác — nếu lo, chạy từng cell một và giữ tab hoạt động).

In [ ]:
# T8.3 — hai loss, cùng seed mặc định
!python -m vietjobs.dl.train_dl --task category --head rnn --loss ce    --device cuda --run-id dl-cat-rnn-ce

In [ ]:
!python -m vietjobs.dl.train_dl --task category --head rnn --loss focal --device cuda --run-id dl-cat-rnn-focal

In [ ]:
# T8.4 — lương, họ cột masked (bài lương tự chọn đúng cache qua family(task))
!python -m vietjobs.dl.train_dl --task salary --head rnn --device cuda --run-id dl-sal-rnn

In [ ]:
# T8.5 — ablation nhánh. "both" đã có sẵn ở dl-cat-rnn-ce, chỉ cần thêm gru/lstm riêng.
!python -m vietjobs.dl.train_dl --task category --head rnn --branches gru  --device cuda --run-id dl-cat-rnn-gru
!python -m vietjobs.dl.train_dl --task category --head rnn --branches lstm --device cuda --run-id dl-cat-rnn-lstm

## 7. Đóng gói kết quả, lưu về Drive

In [ ]:
import shutil

RUN_IDS = ["dl-cat-rnn-ce", "dl-cat-rnn-focal", "dl-sal-rnn", "dl-cat-rnn-gru", "dl-cat-rnn-lstm"]

bundle = pathlib.Path("t8_results")
bundle.mkdir(exist_ok=True)
for r in RUN_IDS:
    shutil.copytree(f"artifacts/{r}", bundle / r, dirs_exist_ok=True)
shutil.copy("docs/04-results.md", bundle / "04-results.md")

zip_path = shutil.make_archive("t8_results", "zip", bundle)
shutil.copy(zip_path, f"{DRIVE_CACHE}/t8_results.zip")
print("đã lưu:", f"{DRIVE_CACHE}/t8_results.zip")

from google.colab import files
files.download(zip_path)   # bản dự phòng, tải trực tiếp qua trình duyệt

## 8. Mang kết quả về máy chính

1. Giải nén `t8_results.zip`.
2. Chép 5 thư mục `dl-cat-rnn-*`, `dl-sal-rnn` vào `artifacts/` của repo cục bộ.
3. Đưa `04-results.md` trong zip cho tôi (hoặc dán 5 dòng mới) — `docs/04-results.md`
   ở máy chính là chỉ-thêm, tôi sẽ ghép đúng 5 dòng mới vào cuối, không sửa dòng cũ.
4. Tôi cập nhật `docs/06-baseline-dl.md`, bảng T8.5, và chương 4 luận văn từ
   `metrics.json`/`history.jsonl` thật trong mỗi thư mục — không chép số tay.